# Chapter 6.5: Traffic Sign Classifier

Back to the course site: https://purwar-lab.github.io/ml-for-robotics-

This notebook trains a convolutional neural network to classify German traffic signs from the GTSRB dataset. Run the cells from top to bottom in Google Colab.

## Cell 1 - Download the GTSRB dataset from Kaggle

Before running this cell, create a Kaggle API token from your Kaggle account page. Paste the full token JSON text when Colab asks for it.

In [ ]:
!pip -q install kaggle

import json
import os
from getpass import getpass

os.makedirs("/root/.kaggle", exist_ok=True)
token_text = getpass("Paste your Kaggle API token JSON text: ").strip()
token = json.loads(token_text)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(token, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -q -o gtsrb-german-traffic-sign.zip -d gtsrb

## Cell 2 - Explore the dataset

This checks the training CSV and shows how many images exist for each of the 43 sign classes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

train_csv = Path("gtsrb/Train.csv")
df = pd.read_csv(train_csv)

display(df.head())
df["ClassId"].value_counts().sort_index().plot(kind="bar", figsize=(14, 4))
plt.title("Images per traffic sign class")
plt.xlabel("Class ID")
plt.ylabel("Image count")
plt.show()

## Cell 3 - Preprocess images with OpenCV

OpenCV loads images as BGR, so this cell converts them to RGB, resizes each image to 32x32, and normalizes pixel values to the 0-1 range.

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

images, labels = [], []

for _, row in df.iterrows():
    path = Path("gtsrb") / row["Path"]
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (32, 32))
    img = img.astype("float32") / 255.0
    images.append(img)
    labels.append(row["ClassId"])

X = np.array(images)
y = np.array(labels)

print(X.shape, y.shape)

## Cell 3b - Define data augmentation

Augmentation creates small image variations during training. Horizontal flips can change the meaning of some traffic signs, so use them with care.

In [ ]:
augmenter = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    horizontal_flip=True,
)

## Cell 4 - Split into train, validation, and test sets

Training data updates the model, validation data helps monitor training, and test data is held back for the final score.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## Cell 5 - Build the CNN architecture

Convolution layers learn visual features, pooling layers shrink the image, and dense layers make the final class decision.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(43, activation="softmax"),
])

model.summary()

## Cell 6 - Compile the model

`adam` updates weights, the loss measures wrongness, and accuracy reports the percent correct.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Cell 7 - Train for 15 epochs

Each epoch passes through the training set once and reports training and validation accuracy.

In [ ]:
history = model.fit(
    augmenter.flow(X_train, y_train, batch_size=64),
    validation_data=(X_val, y_val),
    epochs=15,
)

## Cell 8 - Plot accuracy and loss curves

If training accuracy rises but validation accuracy stalls, the model may be overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].legend()

plt.show()

## Cell 9 - Evaluate on the test set

Use the test set once at the end to estimate real-world performance.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")

## Cell 10 - Confusion matrix and top mistakes

The full matrix shows every class-to-class error, and the sorted list highlights the 10 most common confusions.

In [ ]:
from sklearn.metrics import confusion_matrix

pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(9, 9))
plt.imshow(cm, cmap="Blues")
plt.title("43-class confusion matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.colorbar()
plt.show()

mistakes = []
for true_id in range(43):
    for pred_id in range(43):
        if true_id != pred_id and cm[true_id, pred_id] > 0:
            mistakes.append((cm[true_id, pred_id], true_id, pred_id))

print(sorted(mistakes, reverse=True)[:10])

## Cell 11 - Test one image

This shows a concrete prediction instead of only aggregate metrics.

In [ ]:
idx = 0
probs = model.predict(X_test[idx:idx + 1])[0]
pred_label = np.argmax(probs)
confidence = probs[pred_label]

plt.imshow(X_test[idx])
plt.title(f"Predicted {pred_label}, true {y_test[idx]}, confidence {confidence:.1%}")
plt.axis("off")
plt.show()

## Cell 12 - OpenCV raw-image prediction overlay

This loads a raw sign image with OpenCV, preprocesses it exactly like the training images, feeds it to the CNN, and draws the prediction on top.

In [ ]:
sample_path = Path("gtsrb") / df.iloc[0]["Path"]
raw_bgr = cv2.imread(str(sample_path))
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

model_input = cv2.resize(raw_rgb, (32, 32)).astype("float32") / 255.0
probs = model.predict(model_input[None, ...])[0]
pred_label = np.argmax(probs)
confidence = probs[pred_label]

overlay = raw_rgb.copy()
cv2.putText(
    overlay,
    f"Class {pred_label}: {confidence:.1%}",
    (5, 25),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.7,
    (255, 255, 255),
    2,
)

plt.imshow(overlay)
plt.axis("off")
plt.show()

## Cell 13 - Challenge

Add a third `Conv2D` block before `Flatten`, retrain, and compare validation accuracy and training time.

In [ ]:
# Add before Flatten in Cell 5, then rerun Cells 5-9:
# layers.Conv2D(128, (3, 3), activation="relu"),
# layers.MaxPooling2D((2, 2)),
#
# Write your comparison notes here.